In [1]:
%matplotlib QtAgg
import numpy as np
import pandas as pd
import random as random
import copy
import matplotlib.pyplot as plt


class Capa:
    w : np.ndarray
    y: np.ndarray
    delta: np.ndarray

    def __init__(self, w_i, y_i, delta_i):
        self.w = w_i
        self.y = y_i
        self.delta = delta_i

    def mostrar(self):
        print(f"Pesos: {self.w}")
        print(f"Salidas: {self.y}")
        print(f"Deltas: {self.delta}\n")


def sigm(x):
    return (2/(1+np.exp(-x))) - 1

entrada_usuario = [8,1]

tabla = pd.read_csv('../../Data/gtp_2/concent_trn.csv', header=None).to_numpy()
x0 = -np.ones(len(tabla))
entradas = np.c_[x0, tabla[:,:-1]] # indice -1 := ultima columna ( Acceso a indices con : es [) )
yd =  tabla[:, -1]

print(entradas.shape)
print(yd.shape)

w = np.random.rand(entrada_usuario[0], len(entradas[0])) - 0.5 # dimension: 0 == columnas, dimension: 1 == filas
y_init = np.zeros(entrada_usuario[0])
delta = np.zeros(entrada_usuario[0])
cap = Capa(w,y_init,delta)
vect_capas = [copy.deepcopy(cap)]

# Iniciar red (aleatorio):
for i in range(1,len(entrada_usuario)):
    w = np.random.rand(entrada_usuario[i], entrada_usuario[i-1]+1) - 0.5
    y_init = np.zeros(entrada_usuario[i])
    delta = np.zeros(entrada_usuario[i])
    cap = Capa(w,y_init,delta)
    vect_capas.append(copy.deepcopy(cap))


# Visualizar red inicial:
print(f"Cantidad de capas: {len(vect_capas)}\n")
print(f"Red neuronal: \n")
i = 1
for capa in vect_capas:
    print(f"Capa: {i}")
    capa.mostrar()
    i += 1


(1499, 3)
(1499,)
Cantidad de capas: 2

Red neuronal: 

Capa: 1
Pesos: [[ 2.30008628e-01 -1.38037931e-01  3.37117693e-01]
 [ 4.73250527e-01 -1.41387360e-01 -2.75191763e-01]
 [-1.80491245e-01  3.58660660e-01  3.00033132e-01]
 [-3.74470857e-01  3.70488891e-05  2.64000652e-02]
 [-2.81540303e-03  4.72844979e-01  4.32022114e-01]
 [ 1.99938253e-01  3.96528377e-01 -4.07216268e-01]
 [ 2.14741922e-02 -8.36618717e-02 -3.79758635e-02]
 [-2.79608153e-01  1.76831916e-01 -4.70965411e-02]]
Salidas: [0. 0. 0. 0. 0. 0. 0. 0.]
Deltas: [0. 0. 0. 0. 0. 0. 0. 0.]

Capa: 2
Pesos: [[-0.33640732 -0.4596973   0.09572953  0.16949561 -0.41398074  0.25039979
   0.38383075  0.21882205  0.2081796 ]]
Salidas: [0.]
Deltas: [0.]



In [2]:
# Iterar sobre la red:

epoca = 1
epoca_max = 1000
mu = 0.8

# n -> ejemplo actual
# i -> la capa
# j -> la neurona
while epoca < epoca_max: 

    for n in range(len(entradas)):

        # paso hacia adelante
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                if i==0:
                    z = np.dot(entradas[n,:],vect_capas[i].w[j,:])
                else: 
                    ent = np.r_[-1,vect_capas[i-1].y]
                    z = np.dot(ent,vect_capas[i].w[j,:])
                vect_capas[i].y[j] = sigm(z)

        # propagacion hacia atras
        for i in range(len(vect_capas)-1,-1,-1):
            for j in range(len(vect_capas[i].y)):

                if i==len(vect_capas)-1:
                    vect_capas[i].delta[j] = (1/2) * (yd[n] - vect_capas[i].y[j]) * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])
                else:
                    vect_capas[i].delta[j] = (1/2) * np.dot(vect_capas[i+1].delta, vect_capas[i+1].w[:,j+1])  * (1 + vect_capas[i].y[j]) * (1 - vect_capas[i].y[j])

        #actualizar los pesos
        for i in range(len(vect_capas)):
            for j in range(len(vect_capas[i].y)):
                for m in range(len(vect_capas[i].w[j])):
                    if i==0:
                        vect_capas[i].w[j,m] += mu*vect_capas[i].delta[j]*entradas[n,m]
                    else:
                        ent = np.r_[-1, vect_capas[i-1].y]
                        vect_capas[i].w[j,m] += mu*vect_capas[i].delta[j]*ent[m]
                        
    # Verificación:
    acierto = 0

    for n in range(len(entradas)):
        for capa in range(len(vect_capas)):
            
            for neuron in range(len(vect_capas[capa].y)):
                if capa==0:
                    z = np.dot(entradas[n,:],vect_capas[capa].w[neuron,:])
                else:                
                    ent = np.r_[-1,vect_capas[capa-1].y]
                    z = np.dot(ent,vect_capas[capa].w[neuron,:])
                
                vect_capas[capa].y[neuron] = sigm(z)


        salida_red = vect_capas[-1].y[-1]

        if ((salida_red > 0.0 and yd[n] == 1) or (salida_red < 0.0 and yd[n] == -1)):
            acierto += 1

    tasa_acierto = acierto/len(entradas)
    # print(f"Fin entrenamiento. Epoca: {epoca}, Tasa de acierto: {tasa_acierto * 100: .2f}\n")
    print(f"{tasa_acierto}")

    if tasa_acierto > 0.80 and mu == 0.8:
        mu = 0.5
    elif tasa_acierto >= 0.92 and mu == 0.5:
        mu = 0.3
    elif tasa_acierto >= 0.96 and mu == 0.3:
        mu = 0.1
    elif tasa_acierto >= 0.98 and mu == 0.1:
        mu = 0.05
    elif tasa_acierto >= 0.9889:
        print(f"Convergencia. Epoca {epoca}, Tasa aciertos: {tasa_acierto}")
        break

    # if (tasa_acierto>=0.8 and mu==0.8):
    #     mu = 0.5
    # if (tasa_acierto>=0.92 and mu==0.5):
    #     mu = 0.3
    # if (tasa_acierto>=0.96 and mu==0.3):
    #     mu = 0.1
    # if (tasa_acierto>=0.98 and mu==0.1):
    #     mu = 0.05
    
    epoca += 1





0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.6317545030020013
0.45963975983989325
0.5183455637091394
0.5323549032688459
0.5416944629753169
0.6997998665777185
0.7131420947298199
0.715143428952635
0.7124749833222148
0.7958639092728486
0.8278852568378919
0.8939292861907938
0.8865910607071381
0.8972648432288192
0.914609739826551
0.93862575050

### Test

In [3]:
tabla = pd.read_csv('../../Data/gtp_2/concent_tst.csv', header=None).to_numpy()
x0 = np.ones(len(tabla))*-1
entradas = np.c_[x0,tabla[:,:-1]]
yd =  tabla[:,-1]


plt.ion()
fig, ax = plt.subplots(figsize=(7, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Clasificación: Aciertos y Errores")
ax.set_xlabel("x1")
ax.set_ylabel("x2")

aciertos = 0
for n in range(len(entradas)):
    for i in range(len(vect_capas)):
        for j in range(len(vect_capas[i].y)):
            if i == 0:
                z = np.dot(entradas[n, :], vect_capas[i].w[j, :])
            else: 
                ent = np.r_[-1, vect_capas[i-1].y]
                z = np.dot(ent, vect_capas[i].w[j, :])
            vect_capas[i].y[j] = sigm(z)
    prediccion = vect_capas[-1].y[-1]
    
    x1_val = entradas[n, 1]
    x2_val = entradas[n, 2]

    if prediccion > 0 and yd[n] > 0:
        ax.scatter(x1_val, x2_val, c='k', marker='x', s=40)
        aciertos += 1 
    elif prediccion < 0 and yd[n] < 0:
        ax.scatter(x1_val, x2_val, edgecolors='r', facecolors='None', marker='s', s=40)
        aciertos += 1  
    elif prediccion > 0 and yd[n] < 0:
        ax.scatter(x1_val, x2_val, c='g', marker='x', s=40)
    elif prediccion < 0 and yd[n] > 0:
        ax.scatter(x1_val, x2_val, edgecolors='g', facecolors='None', marker='s', s=40)

plt.ioff()
plt.show()

tasa_aciertos = aciertos / len(entradas)
print(f"Tasa de aciertos: {tasa_aciertos * 100:.2f}%")


Tasa de aciertos: 98.30%
